# 5. Classification - Naive Baseline (Majority Class Classifier)

**Task**: Predict next-hour price direction (`target_class`: 1 = Up, 0 = Down)  
**Baseline**: Always predict the majority class **Up (1)** — since Up = 51.3% of all labels.  
This gives ~51.3% accuracy for free. Every real classifier must beat this.

**Metrics computed**: Accuracy, Precision, Sensitivity (Recall), Specificity, TNR, F1-Score, ROC-AUC, Confusion Matrix  
**Results saved to**: `results/classification_results.csv`

## 1. Imports, Setup & Shared Utilities

The `compute_metrics` and `save_results` functions below are **identical across all classification notebooks** they ensure every model is evaluated on the same 8 metrics and results are appended to the same CSV.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, roc_curve)

# Paths
BASE_DIR     = os.path.abspath(os.path.join(os.getcwd(), '..'))
TRAIN_FILE   = os.path.join(BASE_DIR, 'data', 'final_training_data_classification_train.csv')
TEST_FILE    = os.path.join(BASE_DIR, 'data', 'final_training_data_classification_test.csv')
FEATURES_FILE= os.path.join(BASE_DIR, 'selected_features_classification.json')
SCALER_FILE  = os.path.join(BASE_DIR, 'models', 'scaler_classification.pkl')
RESULTS_DIR  = os.path.join(BASE_DIR, 'results')
PLOTS_DIR    = os.path.join(BASE_DIR, 'plots')
RESULTS_CSV  = os.path.join(RESULTS_DIR, 'classification_results.csv')

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

# ── Shared utility: compute all 8 classification metrics ──────────────────────
def compute_metrics(model_name, y_true, y_pred, y_prob=None):
    """
    Compute Accuracy, Precision, Sensitivity, Specificity, TNR, F1, ROC-AUC.
    y_prob: predicted probabilities for class 1 (needed for ROC-AUC).
            If None (e.g. hard classifiers), AUC is set to NaN.
    Returns a dict and prints a summary table.
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()

    accuracy    = (tp + tn) / (tp + tn + fp + fn)
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0.0   # Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0   # TNR
    tnr         = specificity
    f1          = 2 * precision * sensitivity / (precision + sensitivity) if (precision + sensitivity) > 0 else 0.0
    auc         = roc_auc_score(y_true, y_prob) if y_prob is not None else float('nan')

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"  Confusion Matrix:  TN={tn}  FP={fp}  FN={fn}  TP={tp}")
    print(f"  Accuracy          : {accuracy:.4f}")
    print(f"  Precision         : {precision:.4f}")
    print(f"  Sensitivity/Recall: {sensitivity:.4f}")
    print(f"  Specificity (TNR) : {specificity:.4f}")
    print(f"  F1-Score          : {f1:.4f}")
    print(f"  ROC-AUC           : {auc:.4f}")
    print(f"{'='*55}")

    return {
        'model'      : model_name,
        'accuracy'   : round(accuracy, 4),
        'precision'  : round(precision, 4),
        'sensitivity': round(sensitivity, 4),
        'specificity': round(specificity, 4),
        'tnr'        : round(tnr, 4),
        'f1'         : round(f1, 4),
        'roc_auc'    : round(auc, 4) if not np.isnan(auc) else None,
        'tp': int(tp), 'tn': int(tn), 'fp': int(fp), 'fn': int(fn)
    }

# ── Shared utility: append results row to CSV ─────────────────────────────────
def save_results(metrics_dict):
    row = pd.DataFrame([metrics_dict])
    if os.path.exists(RESULTS_CSV):
        existing = pd.read_csv(RESULTS_CSV)
        # Replace row if model already exists (re-run safety)
        existing = existing[existing['model'] != metrics_dict['model']]
        combined = pd.concat([existing, row], ignore_index=True)
    else:
        combined = row
    combined.to_csv(RESULTS_CSV, index=False)
    print(f"Results saved → {RESULTS_CSV}")

# ── Shared utility: plot confusion matrix ─────────────────────────────────────
def plot_confusion_matrix(y_true, y_pred, model_name, save_path):
    cm = confusion_matrix(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Down (0)', 'Up (1)'],
                yticklabels=['Down (0)', 'Up (1)'], ax=ax)
    ax.set_xlabel('Predicted', fontsize=11)
    ax.set_ylabel('Actual', fontsize=11)
    ax.set_title(f'Confusion Matrix — {model_name}', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Plot saved: {save_path}")

print("Utilities loaded.")

Utilities loaded.


## 2. Load Data

In [2]:
import json

train_df = pd.read_csv(TRAIN_FILE)
test_df  = pd.read_csv(TEST_FILE)

with open(FEATURES_FILE) as f:
    features = json.load(f)

X_train = train_df[features].values
y_train = train_df['target_class'].values
X_test  = test_df[features].values
y_test  = test_df['target_class'].values

print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print(f"Features : {features}")
print(f"Class balance (test) — Up: {y_test.mean()*100:.1f}%  Down: {(1-y_test.mean())*100:.1f}%")

Train: (60299, 10)  |  Test: (10642, 10)
Features : ['stoch_k', 'close_position', 'return_lag_1h', 'log_return_1h', 'log_return_12h', 'volume_change_pct', 'log_return_3h', 'hl_range_pct', 'macd_signal', 'atr_lag_3h']
Class balance (test) — Up: 50.2%  Down: 49.8%


## 3. Majority Class Baseline

**Rule**: Always predict **Up (1)** — the majority class in the training set.  
Since Up = 51.3% of labels, a model that does nothing smarter than this earns ~51.3% accuracy.  
Note: ROC-AUC = 0.5 for any constant predictor (no discrimination ability).

In [ ]:
# Majority class in training set
majority_class = int(np.bincount(y_train).argmax())
print(f"Majority class (from training set): {majority_class} ({'Up' if majority_class == 1 else 'Down'})")

# Prediction: always predict majority class
y_pred_baseline = np.full(len(y_test), majority_class, dtype=int)

# For ROC-AUC with a constant predictor: assign probability = 1.0 for all
# AUC will be 0.5 (no discrimination) — we compute it explicitly to show this
y_prob_baseline = np.full(len(y_test), 1.0)   # constant prob → AUC = 0.5

metrics = compute_metrics(
    model_name='Majority Class Baseline',
    y_true=y_test,
    y_pred=y_pred_baseline,
    y_prob=y_prob_baseline
)

# Note on Specificity:
# Since we always predict Up (1), we never predict Down.
# → TN = 0, so Specificity = TN / (TN + FP) = 0
# This will be immediately visible in the confusion matrix.
print("\nNote: Specificity = 0 because the model never predicts Down.")
print("Any real model must improve BOTH Sensitivity and Specificity.")

In [ ]:
# Confusion matrix plot
plot_path = os.path.join(PLOTS_DIR, 'cm_majority_class_baseline.png')
plot_confusion_matrix(y_test, y_pred_baseline, 'Majority Class Baseline', plot_path)

# Save results
save_results(metrics)